[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TianshuangQiu/TorchCode/blob/master/templates/44_ranks_within_groups.ipynb)

# 🟡 Medium: Ranks Within Groups

Given `group_ids` of shape `(N,)`, return `ranks` of shape `(N,)` where `ranks[i]` is the **position of element `i` within its group**, in the order it appears in the input (0-indexed).

This is the SQL `ROW_NUMBER() OVER (PARTITION BY group)` operation — used for position-within-document in packed sequences, position-within-episode in RL, and de-duplication.

### Signature
```python
def ranks_within_groups(group_ids: torch.Tensor) -> torch.Tensor:
    ...
```

### Rules
- Do **NOT** use Python `for` loops
- `group_ids` may have any non-negative integer values (not necessarily contiguous)

### Example
```
group_ids = [2, 0, 2, 1, 0, 2]
ranks:      [0, 0, 1, 0, 1, 2]
#  group 2 appears at positions 0,2,5 → ranks 0,1,2
#  group 0 appears at positions 1,4   → ranks 0,1
#  group 1 appears at position 3      → rank 0
```

> **Reduction step (say this before coding):** `ranks[i]` = number of elements before position `i` that belong to the same group as `i`. This is a running count — which primitive gives a running count?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def ranks_within_groups(group_ids: torch.Tensor) -> torch.Tensor:
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
group_ids = torch.tensor([2, 0, 2, 1, 0, 2])
result = ranks_within_groups(group_ids)
print('group_ids:', group_ids.tolist())
print('ranks:    ', result.tolist())
print('expected: ', [0, 0, 1, 0, 1, 2])

# Non-contiguous ids
print('non-contiguous [100,100,50]:', ranks_within_groups(torch.tensor([100, 100, 50])).tolist())
print('expected: [0, 1, 0]')

In [ ]:
import torch, time

# ── Test 1: spec example ──────────────────────────────────────────────────
group_ids = torch.tensor([2, 0, 2, 1, 0, 2])
result = ranks_within_groups(group_ids)
expected = torch.tensor([0, 0, 1, 0, 1, 2])
assert result.shape == expected.shape, f"Shape: {result.shape}"
assert torch.equal(result, expected), f"Got {result.tolist()}, expected {expected.tolist()}"
print("Test 1 passed: spec example")

# ── Test 2: all same group → ascending ranks ──────────────────────────────
group_ids = torch.tensor([3, 3, 3, 3])
result = ranks_within_groups(group_ids)
expected = torch.tensor([0, 1, 2, 3])
assert torch.equal(result, expected), f"Got {result.tolist()}"
print("Test 2 passed: all same group")

# ── Test 3: all distinct groups → all rank 0 ─────────────────────────────
group_ids = torch.tensor([5, 2, 7, 1])
result = ranks_within_groups(group_ids)
expected = torch.zeros(4, dtype=torch.long)
assert torch.equal(result, expected), f"Got {result.tolist()}"
print("Test 3 passed: all distinct groups")

# ── Test 4: non-contiguous group IDs ─────────────────────────────────────
group_ids = torch.tensor([100, 100, 50])
result = ranks_within_groups(group_ids)
expected = torch.tensor([0, 1, 0])
assert torch.equal(result, expected), f"Got {result.tolist()}"
print("Test 4 passed: non-contiguous IDs")

# ── Test 5: ranks are 0-indexed and non-negative ──────────────────────────
torch.manual_seed(42)
group_ids = torch.randint(0, 5, (20,))
result = ranks_within_groups(group_ids)
assert result.shape == (20,), f"Shape: {result.shape}"
assert (result >= 0).all(), f"Negative ranks found: {result}"
print("Test 5 passed: ranks non-negative")

# ── Test 6: large N=10000 (timing + spot-check) ───────────────────────────
torch.manual_seed(0)
N = 10000
group_ids = torch.randint(0, 100, (N,))
t0 = time.time()
result = ranks_within_groups(group_ids)
elapsed = time.time() - t0
assert result.shape == (N,), f"Shape: {result.shape}"
assert (result >= 0).all(), "Negative rank"
for i in [0, 1, 100, 999]:
    gid = group_ids[i].item()
    expected_rank = (group_ids[:i] == gid).sum().item()
    assert result[i].item() == expected_rank,         f"Rank at {i}: got {result[i].item()}, expected {expected_rank}"
assert elapsed < 2.0, f"Too slow: {elapsed:.2f}s (expected <2s — no loops)"
print(f"Test 6 passed: N=10000 timing ({elapsed:.3f}s)")

print("\nAll tests passed!")
